# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/24pwai0015-max/flyrank-ml-muhammad-arsalan/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type: Ranking / scoring.**

The decision behind my lane is "which pages should a content strategist review first for a
possible refresh?" That's an ordering question, not a yes/no question — it matches the skill
guide's own mapping directly: *"Which ones first?" -> Ranking / scoring*. The output I need is a
priority score per page: sort every page by that score and hand the strategist a ranked list, not
a single prediction for one page in isolation.

Under the hood the score can come from a binary classifier's predicted probability ("is this page
declining?"), which is what notebooks `01` and `02` already do. But the deliverable and the
evaluation are ranking-shaped, not classification-shaped: I only care whether the pages at the
*top* of the list are the right ones, not whether every page in the middle is labeled correctly.
That's why the metric in section 3 is Precision@K, not accuracy.


In [1]:
import os, sys, subprocess
import pandas as pd

# Setup for Colab -- same pattern as w01_research_question.ipynb
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/24pwai0015-max/flyrank-ml-muhammad-arsalan"
REPO_DIR = "flyrank-ml-muhammad-arsalan"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "CSV not found"

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Total pages in this slice:", len(df))
print("Distinct clients:", df["client_id"].nunique())
print("\nA human can't manually review 30,000 pages a week -- ranking them so a strategist")
print("can work top-down, with limited time, is the whole point of scoring instead of just")
print("classifying every page in isolation.")


Working dir: /home/claude/repo


Total pages in this slice: 30000
Distinct clients: 32

A human can't manually review 30,000 pages a week -- ranking them so a strategist
can work top-down, with limited time, is the whole point of scoring instead of just
classifying every page in isolation.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target: `is_declining_label`, defined as `trend_direction == "down"`.**

This comes from real GSC impression counts: `trend_direction` is set by comparing
`impressions_last_30d` to `impressions_prev_30d` for each page, and a page is labeled `down` when
impressions fell more than 20% between those two windows. So the label is grounded in measured
search-console data, not an editor's subjective call.

It is still a **proxy**, not a direct measurement of "needs refresh." A 20%-drop threshold on a
30-day impression comparison is a reasonable stand-in for decline, but it can mis-flag pages with
naturally noisy, low-volume traffic, and it says nothing about *why* a page is declining
(seasonality, a SERP feature change, a competitor outranking it). I'll treat `is_declining_label`
as "observed decline by this specific measured definition," not as ground truth for "this page
needs an editor's attention" -- and I'll say so plainly whenever I report results.

One consequence of this rule follows immediately: `trend_direction` and `trend_pct` can never be
model features, since they define the label. Section 5 shows what happens when that rule is
broken.


In [2]:
print(df["trend_direction"].value_counts())

declining_rate = (df["trend_direction"] == "down").mean()
print(f"\nDeclining rate (proxy positive rate): {declining_rate:.1%}")


trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Declining rate (proxy positive rate): 54.2%


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric: Precision@50** -- the fraction of correct calls in the top 50 pages of the ranked list.

I'm picking @50 because it matches a real constraint: a content strategist has limited review time
each week. Precision@50 answers the question that actually matters for the decision: *of the pages
I have time to look at, how many were actually worth looking at?* It ignores how well the ranking
does on page 4,000, which is irrelevant since nobody will ever scroll that far.

I'm not using plain accuracy: roughly half of this slice is already labeled "down" (see section 2),
so a model that guesses "declining" for everything would score well on accuracy while being useless
for prioritization. I'm not using ROC-AUC alone either -- AUC rewards the whole ranking equally, and
can't tell me whether the *top* of the list, the only part anyone will act on, is any good.

This metric is computable today, on a real baseline, using only the raw data -- see the code cell
below.


In [3]:
import sys
sys.path.append("scripts")
from ml_utils import precision_at_k

y = (df["trend_direction"] == "down").astype(int)

# A naive, non-ML baseline: rank pages "oldest update first" -- a plausible fixed rule
# an editor might use without any modeling at all.
naive_score = df["days_since_last_update"]
naive_p50 = precision_at_k(y, naive_score, 50)

print(f"Naive 'stalest page first' Precision@50: {naive_p50:.3f}")
print(f"Base rate (what a random ranking would score): {y.mean():.3f}")


Naive 'stalest page first' Precision@50: 0.500
Base rate (what a random ranking would score): 0.542


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one content page**, identified by `content_id`, aggregated over a trailing 90-day
window, and tagged with a pseudonymized `client_id` (32 distinct clients in this slice). The
dataframe below is the actual starter slice -- 30,000 rows x 44 columns -- loaded directly from
`data/raw/content_refresh_anonymized.csv`, with no synthetic rows.


In [4]:
print("Shape:", df.shape)

display_cols = [
    "content_id", "client_id", "content_type", "impressions_90d", "sessions_90d",
    "days_since_last_update", "avg_position", "trend_direction",
]
df[display_cols].head(8)


Shape: (30000, 44)


,content_id,client_id,content_type,impressions_90d,sessions_90d,days_since_last_update,avg_position,trend_direction
0,content_304f48230142,client_f369cb89fc,keyword article,3803,17,20,10.6,down
1,content_a1fb4e703a9e,client_4e07408562,keyword article,15320,9,25,20.3,down
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,12581,11,20,36.5,down
3,content_331d6c4de07b,client_19581e27de,keyword article,11751,78,22,6.2,stable
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,19140,145,14,44.0,down
5,content_d4084a4bc775,client_f369cb89fc,keyword article,3970,5,20,8.5,down
6,content_9a34b442b552,client_8722616204,keyword article,20,1,20,7.0,down
7,content_a63219c6e95a,client_19581e27de,keyword article,1724,28,22,21.2,stable


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

"Is this page declining and worth reviewing" depends on many signals moving together --
traffic volume, position, freshness, content type, keyword competitiveness -- and how much each
one matters shifts by content type and by client. `notebooks/01_first_look_and_discovery.ipynb`
already put a number on that gap: a hand-written rule (a short if/else on one or two signals)
reached **Precision@50 = 0.240** on a client-holdout split (pages from one client never appear in
both train and test); a random forest trained on the same features reached **Precision@50 =
0.740** -- roughly 3.1x better -- because it can combine many weak, tangled signals instead of
relying on the one or two thresholds a human guessed at.

"The model wins" isn't automatically the end of the story, though -- rigor means checking *why* a
model does well before trusting the number. `notebooks/02_your_first_readable_model.ipynb` shows
the failure mode directly: a 2-line decision tree that includes `trend_pct` as a feature reaches
**Precision@50 = 1.000**, which sounds incredible until the printed tree shows it's just
re-deriving the +/-20% threshold that defines the label in the first place. That's leakage, not a
real model. The comparison I'm willing to defend -- 0.240 vs. 0.740 -- excludes `trend_direction`
and `trend_pct` entirely and uses a proper client-holdout split, so no client's pages leak between
train and test.


In [5]:
comparison = pd.DataFrame({
    "method": [
        "Hand-written rule",
        "Random forest (client holdout, no label leakage)",
        "2-line tree with trend_pct as a feature",
    ],
    "precision_at_50": [0.240, 0.740, 1.000],
    "source": [
        "notebooks/01_first_look_and_discovery.ipynb",
        "notebooks/01_first_look_and_discovery.ipynb",
        "notebooks/02_your_first_readable_model.ipynb -- leakage, not a real result",
    ],
})
comparison


,method,precision_at_50,source
0,Hand-written rule,0.24,notebooks/01_first_look_and_discovery.ipynb
1,"Random forest (client holdout, no label leakage)",0.74,notebooks/01_first_look_and_discovery.ipynb
2,2-line tree with trend_pct as a feature,1.00,notebooks/02_your_first_readable_model.ipynb -...


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.